<a href="https://colab.research.google.com/github/thahakm/ml-projects/blob/main/california_housing_decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import fetch_california_housing
house= fetch_california_housing() # Call the function to get the datase
print(house.data.shape) # Access the shape of the data component
print(house.feature_names) # Access the feature names

(20640, 8)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [2]:
x = house.data
y = house.target
y.shape
x.shape

(20640, 8)

In [3]:
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest =train_test_split(x,y,
                                            test_size=0.2 ,
                                            random_state=42)


In [4]:
import pandas as pd
pd.Series(ytrain).value_counts()
# note needed resambling

,count
5.00001,786
1.37500,93
1.62500,89
1.12500,82
1.87500,73
...,...
2.74800,1
4.29000,1
3.10500,1
4.18400,1


In [5]:
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor(random_state=42)
model.fit(xtrain,ytrain)

DecisionTreeRegressor(random_state=42)

In [6]:
ypred = model.predict(xtest)

In [7]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(ytest,model.predict(xtest))

print(f"mean squared error is {mse}")


mean squared error is 0.495235205629094


In [8]:
from sklearn.metrics import mean_squared_error, r2_score

# Check both train and test sets
train_mse = mean_squared_error(ytrain, model.predict(xtrain))
test_mse = mean_squared_error(ytest, model.predict(xtest))

train_r2 = r2_score(ytrain, model.predict(xtrain))
test_r2 = r2_score(ytest, model.predict(xtest))

print(f"Train MSE: {train_mse:.4f}  |  Train R²: {train_r2:.4f}")
print(f"Test MSE:  {test_mse:.4f}  |  Test R²:  {test_r2:.4f}")

Train MSE: 0.0000  |  Train R²: 1.0000
Test MSE:  0.4952  |  Test R²:  0.6221


In [9]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Add guardrails to stop the tree from memorizing
pruned_tree = DecisionTreeRegressor(
    max_depth=8,              # Stops tree at 8 levels deep (prevents micro-branches)
    min_samples_leaf=20,      # A decision leaf MUST have at least 20 houses
    min_samples_split=40,     # Don't split a branch unless it has 40+ houses
    random_state=42
)

pruned_tree.fit(xtrain, ytrain)

# 2. Re-evaluate
train_mse = mean_squared_error(ytrain, pruned_tree.predict(xtrain))
test_mse = mean_squared_error(ytest, pruned_tree.predict(xtest))

train_r2 = r2_score(ytrain, pruned_tree.predict(xtrain))
test_r2 = r2_score(ytest, pruned_tree.predict(xtest))

print(f"Pruned Train MSE: {train_mse:.4f}  |  Train R²: {train_r2:.4f}")
print(f"Pruned Test MSE:  {test_mse:.4f}  |  Test R²:  {test_r2:.4f}")

Pruned Train MSE: 0.3415  |  Train R²: 0.7445
Pruned Test MSE:  0.4164  |  Test R²:  0.6823


In [11]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(20,32), dpi=300)
plot_tree(
    pruned_tree,
    max_depth=3,
    feature_names=house.feature_names,
    filled=True,
    rounded=True,
    fontsize=9,
    precision=2,
)

plt.title(
    "California Housing - Pruned Decision Tree (First 3 Levels)",
    fontsize=16,
    weight="bold",
    pad=15,
)
plt.tight_layout()
plt.show()